# Upload IPCC Glossary to Wikibase

Uploads terms from `outputs/glossary.xml` to a Wikibase instance as structured items.

**Workflow:**
```
outputs/glossary.xml  →  [this notebook]  →  Wikibase items
```

**Two-step process:**
1. **Run Section 3 once** (on a fresh Wikibase) to create the required properties. Note the printed IDs and set them in the Configuration cell.
2. **Run Sections 4–6** to upload all 920 terms.

Each glossary term becomes a Wikibase item with:
- **Label** — term name
- **Description** — `Subject, term, tag: [term name]`
- **Alias** — `also_known_as` value (if different from the label)
- **Statement: instance of** → Category (Q1), with source version qualifier + reference URL/date accessed
- **Statement: part of series** → WGI/WGII/WGIII/SR15 series items
- **Statement: Definition (P13)** → full definition text (monolingualtext, up to 2,500 chars), with source version qualifier + reference URL/date accessed

> **LocalSettings.php prerequisite:** The Wikibase monolingualtext length limit must be raised to 2,500 characters. See the [LocalSettings section](#localsettings-prerequisite) at the bottom of the README.


## 1. Install Dependencies

In [ ]:
%pip install -q wikibaseintegrator python-dotenv

## 2. Configuration

Edit the values below to match your Wikibase instance.

| Variable | Default | Notes |
|---|---|---|
| `WIKIBASE_URL` | `http://localhost:8080` | LOCAL dev instance |
| `WB_USER` / `WB_PASSWORD` | `admin` / `` | Bot or admin credentials |
| `PROP_INSTANCE_OF` | `` | Set after running **Section 3** |
| `PROP_PART_OF` | `` | Set after running **Section 3** |
| `PROP_SOURCE_VERSION` | `` | Set after running **Section 3** |
| `PROP_REFERENCE_URL` | `` | Set after running **Section 3** |
| `PROP_DATE_ACCESSED` | `` | Set after running **Section 3** |
| `PROP_DEFINITION` | `` | Set after running **Section 3** |
| `INSTANCE_OF_QID` | `Q1` | QID of the Category item |
| `DRY_RUN` | `False` | Set `True` to preview without writing |
| `LIMIT` | `0` | Process only first N terms (0 = all) |


In [ ]:
import os
import json
import xml.etree.ElementTree as ET
from pathlib import Path

# ── Load .env (optional) ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    _env = Path.cwd()
    while _env != _env.parent:
        if (_env / '.env').exists():
            load_dotenv(_env / '.env')
            print(f'Loaded .env from {_env / ".env"}')
            break
        _env = _env.parent
except ImportError:
    pass

# ── Settings — edit these ─────────────────────────────────────────────────────
WIKIBASE_URL = os.getenv('WIKIBASE_URL', 'http://localhost:8080')
MW_API_URL   = os.getenv('MW_API_URL',   f'{WIKIBASE_URL}/api.php')
SPARQL_URL   = os.getenv('SPARQL_URL',   'http://localhost:9999/bigdata/sparql')
WB_USER      = os.getenv('WB_USER',      'admin')
WB_PASSWORD  = os.getenv('WB_PASSWORD',  '')   # ← set in .env or here

# ── Property IDs ──────────────────────────────────────────────────────────────
# Leave blank on first run — Section 3 will create missing ones and print IDs.
PROP_INSTANCE_OF    = os.getenv('PROP_INSTANCE_OF',    '')
PROP_PART_OF        = os.getenv('PROP_PART_OF',        '')
PROP_SOURCE_VERSION = os.getenv('PROP_SOURCE_VERSION', '')
PROP_REFERENCE_URL  = os.getenv('PROP_REFERENCE_URL',  '')
PROP_DATE_ACCESSED  = os.getenv('PROP_DATE_ACCESSED',  '')
PROP_DEFINITION     = os.getenv('PROP_DEFINITION',     '')

# ── Other settings ────────────────────────────────────────────────────────────
INSTANCE_OF_QID = os.getenv('INSTANCE_OF_QID', 'Q1')   # Category item
DRY_RUN  = False   # set True to preview without writing to Wikibase
LIMIT    = int(os.getenv('LIMIT', '0'))  # 0 = no limit; set e.g. 9 for test runs
XML_PATH = Path('outputs/glossary.xml')

print(f'Wikibase URL        : {WIKIBASE_URL}')
print(f'User                : {WB_USER}')
print(f'PROP_INSTANCE_OF    : {PROP_INSTANCE_OF or "(not set — run Section 3)"}')
print(f'PROP_PART_OF        : {PROP_PART_OF or "(not set — run Section 3)"}')
print(f'PROP_SOURCE_VERSION : {PROP_SOURCE_VERSION or "(not set — run Section 3)"}')
print(f'PROP_REFERENCE_URL  : {PROP_REFERENCE_URL or "(not set — run Section 3)"}')
print(f'PROP_DATE_ACCESSED  : {PROP_DATE_ACCESSED or "(not set — run Section 3)"}')
print(f'PROP_DEFINITION     : {PROP_DEFINITION or "(not set — run Section 3)"}')
print(f'DRY RUN             : {DRY_RUN}')
print(f'LIMIT               : {LIMIT or "none (all terms)"}')
print(f'XML path            : {XML_PATH}')


## 3. Connect & Set Up Properties

Run this section **once** on a fresh Wikibase to create the required properties.
The script is incremental — it only creates properties that are not yet configured.
Copy the printed IDs into the `.env` file or the Configuration cell above, then proceed to Section 4.

Properties created:
| Property | Datatype | Description |
|---|---|---|
| `instance of` | wikibase-item | Class or type this item is an instance of |
| `part of series` | wikibase-item | IPCC report series this term appears in |
| `source version` | string | Title and version of the source dataset |
| `reference URL` | url | URL of the source reference |
| `date accessed` | time | Date the source was accessed |
| `Definition` | monolingualtext | Full IPCC glossary definition text |


In [ ]:
from wikibaseintegrator import WikibaseIntegrator, wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator import datatypes as wbi_datatypes
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists

wbi_config['MEDIAWIKI_API_URL']   = MW_API_URL
wbi_config['SPARQL_ENDPOINT_URL'] = SPARQL_URL
wbi_config['WIKIBASE_URL']        = WIKIBASE_URL

login = wbi_login.Login(user=WB_USER, password=WB_PASSWORD)
wbi   = WikibaseIntegrator(login=login)
print('Connected to Wikibase.')


In [ ]:
def setup_properties(wbi, existing):
    """Create any missing properties; return {label: pid} for new ones."""
    prop_defs = [
        ('instance of',   'wikibase-item',  'Class or type this item is an instance of'),
        ('part of series','wikibase-item',  'IPCC report series this term appears in'),
        ('source version','string',         'Title and version of the source dataset'),
        ('reference URL', 'url',            'URL of the source reference'),
        ('date accessed', 'time',           'Date the source was accessed'),
        ('Definition',    'monolingualtext','Full IPCC glossary definition text'),
    ]
    created = {}
    for label, datatype, description in prop_defs:
        if existing.get(label):
            continue  # already configured — skip
        prop = wbi.property.new()
        prop.labels.set(language='en', value=label)
        prop.descriptions.set(language='en', value=description)
        prop.datatype = datatype
        result = prop.write()
        created[label] = result.id
        print(f"  Created '{label}': {result.id}")
    return created


existing_props = {
    'instance of':   PROP_INSTANCE_OF,
    'part of series':PROP_PART_OF,
    'source version':PROP_SOURCE_VERSION,
    'reference URL': PROP_REFERENCE_URL,
    'date accessed': PROP_DATE_ACCESSED,
    'Definition':    PROP_DEFINITION,
}

if not all(existing_props.values()):
    print('Creating missing properties …')
    created_props = setup_properties(wbi, existing_props)
    if created_props:
        env_map = {
            'instance of':   'PROP_INSTANCE_OF',
            'part of series':'PROP_PART_OF',
            'source version':'PROP_SOURCE_VERSION',
            'reference URL': 'PROP_REFERENCE_URL',
            'date accessed': 'PROP_DATE_ACCESSED',
            'Definition':    'PROP_DEFINITION',
        }
        print('\n✓ Done. Add to .env and re-run Configuration cell:')
        for label, pid in created_props.items():
            print(f'  {env_map[label]}={pid}')
        # Update in-memory variables too
        for label, pid in created_props.items():
            if label == 'instance of':    PROP_INSTANCE_OF    = pid
            if label == 'part of series': PROP_PART_OF        = pid
            if label == 'source version': PROP_SOURCE_VERSION = pid
            if label == 'reference URL':  PROP_REFERENCE_URL  = pid
            if label == 'date accessed':  PROP_DATE_ACCESSED  = pid
            if label == 'Definition':     PROP_DEFINITION     = pid
else:
    print('All properties already configured — proceeding to upload.')


## 4. Parse Glossary XML

In [ ]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    meta_el = root.find('metadata')
    metadata = {
        'source_version': f"{(meta_el.findtext('title') or '').strip()} {(meta_el.findtext('version') or '').strip()}".strip(),
        'source_url':     (meta_el.findtext('source') or '').strip(),
        'date_accessed':  f"+{(meta_el.findtext('date') or '').strip()}T00:00:00Z",
    }

    terms = []
    for term_el in root.find('terms'):
        name       = (term_el.findtext('name')          or '').strip()
        aka        = (term_el.findtext('also_known_as')  or '').strip()
        definition = (term_el.findtext('definition')     or '').strip()
        series = [
            {'qid': ref.get('qid'), 'label': (ref.text or '').strip()}
            for ref in term_el.findall('.//series_ref')
        ]
        terms.append({
            'id':            term_el.get('id'),
            'name':          name,
            'also_known_as': aka,
            'definition':    definition,
            'series':        series,
        })
    return metadata, terms


metadata, terms = parse_xml(XML_PATH)
if LIMIT:
    terms = terms[:LIMIT]
print(f'Parsed {len(terms)} terms from {XML_PATH}')
print(f'Source  : {metadata["source_version"]}')
print(f'URL     : {metadata["source_url"]}')
print(f'Date    : {metadata["date_accessed"]}')
print(f'\nFirst term preview:')
t = terms[0]
print(f'  name          : {t["name"]}')
print(f'  also_known_as : {t["also_known_as"]}')
print(f'  definition    : {t["definition"][:120]}…')
print(f'  series        : {t["series"]}')


## 5. Upload Items

Set `DRY_RUN = True` in the Configuration cell to preview without writing.

> **Note:** Running this cell twice will create duplicate items. Check `outputs/upload_log.json` for QIDs of previously created items.


In [ ]:
def upload_term(wbi, term, metadata, prop_instance_of, prop_part_of,
                prop_source_version, prop_reference_url, prop_date_accessed,
                prop_definition='', dry_run=False):
    item = wbi.item.new()

    item.labels.set(language='en', value=term['name'])
    item.descriptions.set(language='en', value=f"Subject, term, tag: {term['name']}")

    if term['also_known_as'] and term['also_known_as'] != term['name']:
        item.aliases.set(language='en', values=[term['also_known_as']])

    claims = []

    # instance of: Category (Q1) with source qualifier + reference
    if prop_instance_of:
        qualifiers = Qualifiers()
        if prop_source_version and metadata['source_version']:
            qualifiers.add(wbi_datatypes.String(
                prop_nr=prop_source_version,
                value=metadata['source_version'],
            ))

        references = References()
        ref = Reference()
        if prop_reference_url and metadata['source_url']:
            ref.add(wbi_datatypes.URL(prop_nr=prop_reference_url, value=metadata['source_url']))
        if prop_date_accessed and metadata['date_accessed']:
            ref.add(wbi_datatypes.Time(
                prop_nr=prop_date_accessed,
                time=metadata['date_accessed'],
                precision=11, timezone=0, before=0, after=0,
                calendarmodel='http://www.wikidata.org/entity/Q1985727',
            ))
        if ref.snaks:
            references.add(ref)

        claims.append(wbi_datatypes.Item(
            prop_nr=prop_instance_of,
            value=INSTANCE_OF_QID,
            qualifiers=qualifiers,
            references=references,
        ))

    # part of series
    if prop_part_of:
        for series in term['series']:
            qid = (series.get('qid') or '').strip()
            if qid:
                claims.append(wbi_datatypes.Item(prop_nr=prop_part_of, value=qid))

    # Definition (monolingualtext) with source qualifier + reference
    if prop_definition and term['definition']:
        qualifiers = Qualifiers()
        if prop_source_version and metadata['source_version']:
            qualifiers.add(wbi_datatypes.String(
                prop_nr=prop_source_version,
                value=metadata['source_version'],
            ))

        references = References()
        ref = Reference()
        if prop_reference_url and metadata['source_url']:
            ref.add(wbi_datatypes.URL(prop_nr=prop_reference_url, value=metadata['source_url']))
        if prop_date_accessed and metadata['date_accessed']:
            ref.add(wbi_datatypes.Time(
                prop_nr=prop_date_accessed,
                time=metadata['date_accessed'],
                precision=11, timezone=0, before=0, after=0,
                calendarmodel='http://www.wikidata.org/entity/Q1985727',
            ))
        if ref.snaks:
            references.add(ref)

        claims.append(wbi_datatypes.MonolingualText(
            prop_nr=prop_definition,
            text=term['definition'],
            language='en',
            qualifiers=qualifiers,
            references=references,
        ))

    if claims:
        item.claims.add(claims)

    if dry_run:
        return None

    result = item.write()
    return result.id


In [ ]:
uploaded = []
failed   = []
total    = len(terms)

for i, term in enumerate(terms, 1):
    try:
        qid = upload_term(
            wbi, term, metadata,
            PROP_INSTANCE_OF, PROP_PART_OF,
            PROP_SOURCE_VERSION, PROP_REFERENCE_URL, PROP_DATE_ACCESSED,
            PROP_DEFINITION, dry_run=DRY_RUN,
        )
        label = '[DRY RUN]' if DRY_RUN else qid
        uploaded.append({'term': term['name'], 'qid': qid, 'series': [s['qid'] for s in term['series']]})
        print(f'[{i:>4}/{total}] ✓  {term["name"]}  →  {label}')
    except Exception as exc:
        failed.append({'term': term['name'], 'error': str(exc)})
        print(f'[{i:>4}/{total}] ✗  {term["name"]}  ERROR: {exc}')

print(f'\n── Summary {"(DRY RUN) " if DRY_RUN else ""}──────────────────────')
print(f'  Uploaded : {len(uploaded)}')
print(f'  Failed   : {len(failed)}')


## 6. Save Results & Review


In [ ]:
import IPython.display as display
import html as html_lib

log_path = Path('outputs/upload_log.json')
with open(log_path, 'w', encoding='utf-8') as fh:
    json.dump(
        {'uploaded': uploaded, 'failed': failed
        fh, indent=2, ensure_ascii=False,
    )
print(f'Log saved to {log_path}')

if failed:
    print('\nFailed terms:')
    for f in failed:
        print(f'  • {f["term"]}: {f["error"]}')
else:
    print('All terms uploaded successfully.')

# Quick summary table (first 20)
rows = ''.join(
    f'<tr><td>{html_lib.escape(r["term"])}</td><td>{r["qid"] or "DRY RUN"}</td></tr>'
    for r in uploaded[:20]
)
truncated = f'<p><em>… and {len(uploaded)-20} more (see upload_log.json)</em></p>' if len(uploaded) > 20 else ''
display.display(display.HTML(
    f'<table><thead><tr><th>Term</th><th>QID</th></tr></thead><tbody>{rows}</tbody></table>{truncated}'
))
